In [5]:
import numpy.random as rand
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge as ridge
from sklearn.linear_model import Lasso as lasso
from sklearn.linear_model import LinearRegression as ols


from sklearn.ensemble import RandomForestRegressor as rfr
from sklearn.tree import DecisionTreeRegressor as reg_tree
from sklearn.ensemble import AdaBoostRegressor as ada_reg
from sklearn.ensemble import GradientBoostingRegressor as gbr
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import train_test_split
import copy

import matplotlib
from sklearn.metrics import mean_squared_error as mse

from tqdm import tqdm
import pandas as pd

In [2]:
import rpy2.robjects as ro
readRDS = ro.r['readRDS']

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


In [6]:
def get_best_for_data(X, Y, regs):
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2) # doesn't change X
    val_errs = []
    models = []
    for reg in regs:
        model = copy.deepcopy(reg)
        model.fit(x_train, y_train)
        val_errs.append(mse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model))
    min_ind = val_errs.index(min(val_errs))
    print(str(model)[:40], val_errs[min_ind])
    return copy.deepcopy(models[min_ind])

In [19]:
result_df = pd.DataFrame({"test_X", "omega","eta","pred_HTE","true_HTE"})

for m in tqdm(range(100)):
    path = 'C:/integraion_RCT_and_Obs/integration_RCT_and_Obs/01_data/data/1d_squared_montecarlo/'+str(m+1)+'.obj'
    obj = readRDS(path) 
    obj_kes_list = obj.names 

    train_data = obj[0]
    test_data  = obj[1]

    # train data
    X_full   = np.array(train_data[0])
    Y_full   = np.array(train_data[1])
    Tau_full = np.array(train_data[2])
    T_full   = np.array(train_data[5])
    ID       = np.array(train_data[6])

    # sepalate  RCT and Obs
    X   = X_full[ID == "O"]
    X_E = X_full[ID == "R"]
    Y   = Y_full[ID == "O"]
    Y_E = Y_full[ID == "R"]
    T   = T_full[ID == "O"]
    T_E = T_full[ID == "R"]

    # model
    regs = [rfr(n_estimators=i) for i in [10, 20, 40, 60, 100, 150, 200]]
    regs += [reg_tree(max_depth=i) for i in [5, 10, 20, 30, 40, 50]]
    regs += [ada_reg(n_estimators=i) for i in [10, 20, 50, 70, 100, 150, 200]]
    regs += [gbr(n_estimators=i) for i in [50, 70, 100, 150, 200]]

    f1pred_exp = get_best_for_data(X_E[T_E>0].reshape(-1,1), Y_E[T_E>0], regs)
    f0pred_exp = get_best_for_data(X_E[T_E==0].reshape(-1,1), Y_E[T_E==0], regs)
    f1pred_obs = get_best_for_data(X[T>0].reshape(-1,1), Y[T>0], regs)
    f0pred_obs = get_best_for_data(X[T==0].reshape(-1,1), Y[T==0], regs)

    omega = f1pred_obs.predict(X_E.reshape(-1,1)) - f0pred_obs.predict(X_E.reshape(-1,1)) 
    tau = f1pred_exp.predict(X_E.reshape(-1,1)) - f0pred_exp.predict(X_E.reshape(-1,1))

    omega_O = f1pred_obs.predict(X.reshape(-1,1)) - f0pred_obs.predict(X.reshape(-1,1)) 
    tau_O = f1pred_exp.predict(X.reshape(-1,1)) - f0pred_exp.predict(X.reshape(-1,1))

    eta_est = tau - omega
    eta_est_O = tau_O - omega_O

    eta_ridge = get_best_for_data(X_E.reshape(-1, 1), eta_est, [ridge(alpha=a) for a in [1e-10]])

    # inference
    tau   = Tau_full[ID == "O"]
    tau_E = Tau_full[ID == "R"]

    test_X = np.array(test_data[0] )
    true_test_tau = np.array(test_data[1])

    omega =  f1pred_obs.predict(test_X) - f0pred_obs.predict(test_X)
    eta   = eta_ridge.predict(test_X)

    # save 
    i_result = pd.DataFrame({"test_X": test_X.reshape(-1),
                             "omega": omega,
                             "eta": eta,
                             "pred_HTE": omega + eta,
                             "true_HTE": true_test_tau.reshape(-1)})
    result_df = pd.concat([result_df, i_result])

  0%|          | 0/100 [00:00<?, ?it/s]

GradientBoostingRegressor(n_estimators=2 1.1413468128062687
GradientBoostingRegressor(n_estimators=2 1.53481507840799
GradientBoostingRegressor(n_estimators=2 1.129107602409215


  1%|          | 1/100 [00:02<04:31,  2.75s/it]

GradientBoostingRegressor(n_estimators=2 1.3399351781132138
Ridge(alpha=1e-10) 1.8400724878959103
GradientBoostingRegressor(n_estimators=2 1.362646283898054
GradientBoostingRegressor(n_estimators=2 0.41161799685328954
GradientBoostingRegressor(n_estimators=2 1.4042239380190122


  2%|▏         | 2/100 [00:05<04:15,  2.61s/it]

...

GradientBoostingRegressor(n_estimators=2 1.0952556896775
Ridge(alpha=1e-10) 2.01991503532005
GradientBoostingRegressor(n_estimators=2 0.7783819707160251
GradientBoostingRegressor(n_estimators=2 2.522421971504871
GradientBoostingRegressor(n_estimators=2 1.571540422168093


 99%|█████████▉| 99/100 [05:52<00:02,  2.90s/it]

GradientBoostingRegressor(n_estimators=2 1.514827980018563
Ridge(alpha=1e-10) 1.084711554800299
GradientBoostingRegressor(n_estimators=2 1.5951435187119818
GradientBoostingRegressor(n_estimators=2 0.980549723570338
GradientBoostingRegressor(n_estimators=2 1.8597496687663675


100%|██████████| 100/100 [05:55<00:00,  3.55s/it]

GradientBoostingRegressor(n_estimators=2 1.7772532348316816
Ridge(alpha=1e-10) 1.0458659985299186


In [27]:
result_df = result_df[["test_X", "omega", "eta", "pred_HTE", "true_HTE"]].iloc[5:,]

In [28]:
result_df.to_csv("C:\integraion_RCT_and_Obs/integration_RCT_and_Obs/03_analyze/result/0113\python_montecarlo/kallus.csv", index=False)

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\hazi-\AppData\Local\Temp\ipykernel_35148\516418879.py:1: SyntaxWarning: invalid escape sequence '\i'
  result_df.to_csv("C:\integraion_RCT_and_Obs/integration_RCT_and_Obs/03_analyze/result/0113\python_montecarlo/kallus.csv", index=False)
